In [2]:
## This is a preliminary code to generate questions based on Wikipedia article.
## To do:  generate also answers.

# This first block is not using any wikipedia info, it basically just finds the topic/focus words.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def generate_questions(text):
    model_name = "google/t5-small-ssm"  # You can experiment with different models
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)

    questions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return questions

# Example usage:
text = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars, named after the engineer Gustave Eiffel. It was built for the 1889 World's Fair, when France celebrated the centenary of the French Revolution."

#questions = generate_questions(text)
question = tokenizer.decode(outputs[0], skip_special_tokens=True)
for question in questions:
    print(question)


NameError: name 'tokenizer' is not defined

In [3]:
!pip install wikipedia
!pip install transformers
!pip install torch

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11679 sha256=0adefcff68e45b1c175db8e4ff53efb8c6a624415c26a1a2a98150d43fa55b35
  Stored in directory: /root/.cache/pip/wheels/5e/b6/c5/93f3dec388ae76edc830cb42901bb0232504dfc0df02fc50de
Successfully built wikipedia


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
## Nikolay Vorontsov, 11.11.2024

## references:
## based on Gemini prompts: "Generate questions from Wikipedia articles", and some further manipulations with it, including:
## "and for wikipedia?", "make it print just 1 random question per article"
##
## https://medium.com/@anoopjohny2000/building-an-interactive-question-answering-app-with-streamlit-transformers-and-langchain-13b338cfe534
## https://github.com/paoyw/2022_spring_ML



In [6]:
# the code that generates a random chunk of the Wikipedia article and a question based on that chunk and prints it into a json file.

In [14]:
questions_chunks_pairs = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/jsons/questions_18112024.jsonl"

In [8]:
topics = [
    "The Cultural Impact of Stanley Kubrick's '2001: A Space Odyssey'",
    "The Lost City of Petra: A Hidden Gem of the Nabatean Kingdom",
    "The Enigma of the Voynich Manuscript",
    "The Life and Legacy of Frida Kahlo",
    "The Battle of Gettysburg: A Turning Point in the American Civil War",
    "The Rise and Fall of the Soviet Union",
    "The Psychology of Serial Killers",
    "The Making of \"The Lord of the Rings\" Trilogy",
    "The History of Artificial Intelligence",
    "The Impact of Climate Change on the Great Barrier Reef"
]

topics2 = [
    "The Forgotten Women Mathematicians of the 19th Century",
    "The Secret History of the Knights Templar",
    "The Cultural Impact of the Silent Film Era",
    "The Rise and Fall of the Kingdom of Aksum",
    "The Psychology of Urban Legends",
    "The Forgotten Battles of World War I",
    "The Secret Societies of 19th Century Paris",
    "The Impact of Climate Change on Indigenous Communities",
    "The Rise and Fall of the Ottoman Empire",
    "The Forgotten Explorers of the Arctic"
]

topics3 = [
    "Kubrick",
    "Petra",
    "Voynich",
    "Kahlo",
    "Gettysburg",
    "Soviet",
    "Killers",
    "Rings",
    "AI",
    "Reef"
]


In [15]:
import random
import json
import torch
import wikipedia
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, QuestionAnsweringPipeline

# Specify the device outside the function
device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_question_from_wikipedia(topic, filename=questions_chunks_pairs):
    try:
        page = wikipedia.page(topic)
        text = page.content

        # Get a random chunk of the text
        start_index = random.randint(0, len(text) - 512)
        chunk = text[start_index:start_index + 512]

        # Load the model and tokenizer
        model_name = "deepset/roberta-base-squad2"
        model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Create the question-answering pipeline
        question_generator = QuestionAnsweringPipeline(model=model, tokenizer=tokenizer, device=device)

        # Generate a question based on the chunk
        result = question_generator(question="What are some questions about this text?", context=chunk)
        question = result['answer']

        # Print the question and chunk
        print("Question:", question)
        print("Chunk:", chunk)
        print()

        # Create a dictionary (a JSON object) to store question and chunk
        data = {"question": question, "chunk": chunk}

        # Append the JSON object to the file in JSONL format
        with open(filename, 'a') as f:
            json.dump(data, f, ensure_ascii=False)
            f.write('\n')

        return question, chunk

    except wikipedia.exceptions.DisambiguationError as e:
        print(f"Disambiguation error: {e}")
        return None, None
    except wikipedia.exceptions.PageError as e:
        print(f"Page not found: {e}")
        return None, None

# Example usage:
#topic = "Eiffel Tower"
for topic in topics3:
  question, chunk = generate_question_from_wikipedia(topic)


Question: irrational
Chunk: idered by some critics to be irrational he firmly believed that actors were at their best during filming, as opposed to in rehearsals, saying, "[w]hen you make a movie, it takes a few days just to get used to the crew, because it is like getting undressed in front of fifty people. Once you're accustomed to them, the presence of even one other person on set is discordant and tends to produce self-consciousness in the actors, and certainly in itself".
In 1987, when Kubrick was asked about his reputation for e

Question: 
==== Economy ====
Chunk: ping and it became inscribed within the bourgeois culture.


==== Economy ====
As the popularity of pet-keeping in the modern sense rose during the Victorian era, animals became a fixture within urban culture as commodities and decorative objects. Pet keeping generated a commercial opportunity for entrepreneurs. By the mid-19th century, nearly twenty thousand street vendors in London dealt with live animals. The popula

/usr/local/lib/python3.10/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.10/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [10]:
## The next part of the code will use generative model, to create a proper question from it.


In [11]:
import google.generativeai as genai
#import os

In [16]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') #os.environ.get("API_GEMINI")

genai.configure(api_key=GOOGLE_API_KEY)
LLM = "gemini-1.5-flash"
model = genai.GenerativeModel(LLM)


In [19]:
output_file = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/jsons/questions.corrected_1811204_only_clean.jsonl"

In [20]:
import json

def extract_with_ids(json_file, output_file):
   with open(json_file, 'r') as f, open(output_file, 'w') as out_f:
        for index, line in enumerate(f, start=1):
            data = json.loads(line)
            question = data['question']
            chunk = data['chunk']
            language = "English"

            system_prompt = f"Given a question: ({question}) and an answer: ({chunk}), convert both the question and answer into two separate correct {language} sentences, separate them with a new line symbol, limit the exported question to 10 words and exported answer to 512 characters."

            #messages = []
            #messages.append(system_prompt)

            try:
                r = model.generate_content(system_prompt).text
                print(f"Id: {index}")
                print("Question:", question)
                print("Chunk:", chunk)
                print("Language:", language)
                print("Response:",)
                print(r)
                print("\n")

                response_lines = [line.strip() for line in r.splitlines() if line.strip()]
                if len(response_lines) > 2:
                  response_lines = response_lines[:2]
                question_edited, answer_edited = response_lines

                # Create a JSON object for the result
                result_data = {
                    "id": index,
                    #"question": question,
                    #"chunk": chunk,
                    "question_edited": question_edited,
                    "answer_edited": answer_edited
                    }

                # Write the result to the output file in JSONL format
                json.dump(result_data, out_f)
                out_f.write('\n')

            except Exception as e:
                print(f"Error processing item {index}: {e}")

extract_with_ids(questions_chunks_pairs, output_file)


Id: 1
Question: irrational
Chunk: idered by some critics to be irrational he firmly believed that actors were at their best during filming, as opposed to in rehearsals, saying, "[w]hen you make a movie, it takes a few days just to get used to the crew, because it is like getting undressed in front of fifty people. Once you're accustomed to them, the presence of even one other person on set is discordant and tends to produce self-consciousness in the actors, and certainly in itself".
In 1987, when Kubrick was asked about his reputation for e
Language: English
Response:
Was Stanley Kubrick irrational?

Some critics considered Kubrick irrational; he believed actors performed best during filming, not rehearsals, stating, "When you make a movie, it takes a few days just to get used to the crew...Once you're accustomed to them, the presence of even one other person on set is discordant and tends to produce self-consciousness."



Id: 2
Question: 
==== Economy ====
Chunk: ping and it became i